# Đề tài 2: Phân Đoạn Ảnh Y Tế (Kvasir-SEG) — Môi Trường Google Colab / Kaggle Setup

Notebook thiết lập và xác thực toàn bộ pipeline dữ liệu, kiểm tra phần cứng GPU, clone repository, cài đặt môi trường và kiểm thử trực quan hóa dữ liệu trước khi bước sang giai đoạn huấn luyện U-Net và SegFormer (Tuần 2).

> **Hướng dẫn khởi chạy nhanh trên Google Colab**:
> 1. Vào menu **Runtime** -> **Change runtime type** -> Chọn Hardware accelerator: **T4 GPU** (hoặc GPU bất kỳ).
> 2. Chạy tuần tự các ô lệnh bên dưới bằng phím tắt `Shift + Enter`.


In [ ]:
# 1. Kiểm tra cấu hình phần cứng GPU và phiên bản CUDA
!nvidia-smi

import torch
print("\n--- Kiểm tra PyTorch và CUDA ---")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Cảnh báo: Hiện tại chưa bật GPU Runtime. Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")


## 2. Clone Repository từ GitHub và Thiết lập Thư Mục Làm Việc
Tải toàn bộ mã nguồn của nhóm từ repository [GitHub DL-Group7](https://github.com/daoviet2/DL-Group7.git).


In [ ]:
import os

REPO_URL = "https://github.com/daoviet2/DL-Group7.git"
REPO_NAME = "DL-Group7"

if not os.path.exists(REPO_NAME) and not os.path.exists("configs"):
    print(f"[Git] Đang clone repository từ {REPO_URL} ...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME) and not os.path.exists("configs"):
    %cd {REPO_NAME}
else:
    print("[Git] Đang ở trong thư mục dự án:", os.getcwd())

!git status


## 3. Cài Đặt Các Thư Viện Phụ Thuộc (Dependencies)
Cài đặt các gói thư viện chuẩn cho cả 3 thành viên: Albumentations, Segmentation Models PyTorch, Transformers, OpenCV, v.v.


In [ ]:
# Cài đặt từ requirements.txt
!pip install -r requirements.txt -q

# Kiểm tra nạp các module chính
import cv2
import albumentations as A
import segmentation_models_pytorch as smp
import transformers
import pandas as pd
import numpy as np

print("\n--- Xác nhận các thư viện cốt lõi ---")
print("Albumentations :", A.__version__)
print("OpenCV         :", cv2.__version__)
print("SMP (U-Net)    :", smp.__version__)
print("Transformers   :", transformers.__version__)
print("[OK] Toàn bộ thư viện đã sẵn sàng!")


## 4. Tải Dữ Liệu Kvasir-SEG (1000 Ảnh Kèm Mặt Nạ Phân Đoạn)
Tự động tải bộ dữ liệu từ nguồn chính thức của *Simula Research Laboratory*.
Dữ liệu sẽ được giải nén vào `./Kvasir-SEG` (bao gồm `images/`, `masks/`, `kavsir_bboxes.json`).


In [ ]:
# Tải và giải nén dữ liệu tự động
!python scripts/download_data.py

# Xác nhận số lượng mẫu
import os
img_count = len([f for f in os.listdir("Kvasir-SEG/images") if f.endswith(".jpg")])
mask_count = len([f for f in os.listdir("Kvasir-SEG/masks") if f.endswith(".jpg")])
print(f"\n[Dữ liệu] Images: {img_count}/1000 | Masks: {mask_count}/1000")
assert img_count == 1000 and mask_count == 1000, "Dữ liệu tải chưa đầy đủ 1000 cặp ảnh-mask!"
print("[OK] Bộ dữ liệu Kvasir-SEG đã toàn vẹn.")


## 5. Xác Nhận Danh Sách Chia Tập Phân Tầng (Stratified 80/10/10)
Kiểm tra các file phân tầng cố định `splits/train.txt` (800), `splits/val.txt` (100), `splits/test.txt` (100) và `splits/metadata.csv`.


In [ ]:
# Kiểm tra nếu chưa có splits thì chạy script phân tầng
if not os.path.exists("splits/metadata.csv"):
    !python scripts/prepare_splits.py
else:
    print("[Splits] Thư mục splits/ đã có sẵn từ repo:")
    df_meta = pd.read_csv("splits/metadata.csv")
    print(f"Tổng số mẫu metadata: {len(df_meta)}")
    print(df_meta.groupby(["split", "size_category"]).size().unstack(fill_value=0))


## 6. Chạy Kiểm Thử Tự Động Toàn Vẹn (Sanity Check)
Kiểm tra DataLoader, định dạng Tensor, tính nhị phân của Mask (khử nhiễu JPEG) và tọa độ Bounding Box.


In [ ]:
!python scripts/sanity_check_data.py


## 7. Trực Quan Hóa Tương Tác Trực Tiếp Trên Notebook
Hiển thị một mini-batch từ `train_loader` để trực quan kiểm tra ảnh gốc, ground truth mask và overlay ranh giới polyp.


In [ ]:
import matplotlib.pyplot as plt
from src.datasets.kvasir import get_dataloaders
from src.utils.config import load_config
from src.utils.vis import create_overlay, draw_bounding_box
from src.datasets.transforms import denormalize_image

# Nạp cấu hình
cfg = load_config("configs/base.yaml")
cfg_demo = cfg.copy()
cfg_demo["data"]["num_workers"] = 2
cfg_demo["data"]["batch_size"] = 4

train_loader, val_loader, test_loader = get_dataloaders(cfg_demo)
batch = next(iter(train_loader))

images = batch["image"]
masks = batch["mask"]
sample_ids = batch["id"]
bboxes = batch["bbox"]

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i in range(4):
    img_denorm = denormalize_image(images[i])
    mask_np = masks[i, 0].numpy()
    
    # Vẽ BBox lên ảnh gốc
    img_box = draw_bounding_box(img_denorm, bboxes[i], color=(255, 50, 50), thickness=2)
    # Vẽ Overlay
    overlay = create_overlay(img_denorm, mask_np, color=(0, 230, 80), alpha=0.45)
    
    axes[i, 0].imshow(img_box)
    axes[i, 0].set_title(f"Ảnh gốc (BBox): {sample_ids[i]}", fontsize=11)
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask", fontsize=11)
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Overlay Phân Đoạn", fontsize=11)
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


## 8. Kiểm Thử Khởi Tạo Mô Hình & Forward Pass Trên GPU (Chuẩn Bị Cho Tuần 2)
Thử nghiệm đưa một mini-batch qua mô hình trên GPU để kiểm tra tiêu thụ bộ nhớ VRAM và tốc độ xử lý.


In [ ]:
if torch.cuda.is_available():
    import time
    device = torch.device("cuda")
    
    # Kiểm tra thử một mô hình phân đoạn mẫu từ SMP (ResNet34 U-Net)
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1
    ).to(device)
    
    dummy_input = torch.randn(8, 3, 256, 256, device=device)
    
    start_time = time.time()
    with torch.no_grad():
        output = model(dummy_input)
    elapsed = (time.time() - start_time) * 1000
    
    allocated_mem = torch.cuda.memory_allocated(device) / (1024**2)
    cached_mem = torch.cuda.memory_reserved(device) / (1024**2)
    
    print("\n--- Kiểm Thử Forward Pass GPU Hoàn Tất ---")
    print(f"Shape đầu vào        : {dummy_input.shape}")
    print(f"Shape đầu ra (Logits) : {output.shape}")
    print(f"Thời gian Forward    : {elapsed:.2f} ms")
    print(f"VRAM cấp phát        : {allocated_mem:.2f} MB")
    print(f"VRAM dự trữ          : {cached_mem:.2f} MB")
    print("[OK] Môi trường Colab GPU đã sẵn sàng 100% cho Tuần 2!")
else:
    print("Đang chạy trên CPU. Hãy kích hoạt GPU T4 trên Colab để tăng tốc huấn luyện!")
